## Creating and Working with a Delta Table

In [0]:
%sql
USE CATALOG azuredatabricks;
USE SCHEMA default;

SELECT current_catalog(), current_schema();


#### Describing the Schema

In [0]:
%sql
DESCRIBE SCHEMA EXTENDED default;

#### Show Tables

In [0]:
%sql
SHOW TABLES;

#### Show Volumes


In [0]:
%sql
SHOW VOLUMES;

#### Listing files/data in volumes

In [0]:
spark.sql("LIST '/Volumes/azuredatabricks/default/myfiles'").display()

### Create a Delta Table from a CSV File

In [0]:
spark.sql('''
          SELECT *
          FROM csv.`/Volumes/azuredatabricks/default/myfiles`
          ''').display()

In [0]:
spark.sql('''
          SELECT *
          FROM text.`/Volumes/azuredatabricks/default/myfiles`
          ''').display()

In [0]:
%sql
SELECT *
FROM read_files(
  '/Volumes/azuredatabricks/default/myfiles',
  format => 'csv',
  header => true,
  inferSchema => true
)

In [0]:
%sql
DROP TABLE IF EXISTS current_employees;

CREATE TABLE current_employees AS
SELECT
  ID,
  Firstname,
  Country,
  Role
FROM read_files(
  '/Volumes/azuredatabricks/default/myfiles',
  format => 'csv',
  header => true,
  inferSchema => true
);

-- Display the table
SELECT * FROM current_employees;


#### Reading data in Python

In [0]:
sdf = (spark
       .read
       .format('csv')
       .option('header', True)
       .option('inferSchema', True)
       .load('/Volumes/azuredatabricks/default/myfiles'))
sdf.display()


In [0]:
# Create a Delta Table from Spark Dataframe
(sdf
 .write
 .mode("overwrite")
 .format("delta")
 .saveAsTable('current_employees_py')
);

In [0]:
spark.sql("SELECT * FROM current_employees_py").display()

In [0]:
# Read the Delta Table using Python

(spark
 .read
 .table('current_employees_py')
).display()
 

In [0]:
spark.catalog.listTables("azuredatabricks.default")

In [0]:
%sql
SELECT *
FROM current_employees

In [0]:
%sql
DESCRIBE DETAIL current_employees;

In [0]:
%sql
DESCRIBE EXTENDED current_employees;

In [0]:
%sql
DESCRIBE HISTORY current_employees;

### Insert, Delete and Update Records in the Delta Table


In [0]:
%sql
SELECT *
FROM current_employees;

In [0]:
%sql
-- 1. Insert two employees into the table
INSERT INTO current_employees
VALUES (5555, 'Alex', 'USA', 'Instructor'),
(6666, 'Sanjay', 'India', 'Instrcutor');

-- 2. Update a record in the table
UPDATE current_employees
SET Role = "Senior Manager"
WHERE id = 1111;

-- 3. Delete a record in the table
DELETE FROM current_employees
WHERE ID = 3333;

In [0]:
%sql
SELECT *
FROM current_employees;

In [0]:
%sql
DESCRIBE HISTORY current_employees;

In [0]:
%sql
SELECT *
FROM current_employees VERSION AS OF 0;

In [0]:
%sql
DROP TABLE IF EXISTS current_employees;
DROP TABLE IF EXISTS current_employees_py;